# 03 — LLM-as-judge: scoring what arithmetic cannot reach

## Why a judge at all
Subtraction catches *timing* failures. But "did the agent handle the caller's vague answer gracefully?" or "did it claim something the caller never said?" are **semantic** judgments. Pre-LLM, you needed human raters for every call (slow, expensive). The 2023+ move: use a strong LLM as the rater — **LLM-as-judge** — and then *prove* it agrees with humans on a sample (that proof is notebook 04).

## Our judge contract (each line exists for a reason)
- **temperature 0** — sampling randomness off; same input → same verdict (reproducibility).
- **JSON mode** — output is machine-parseable, goes straight into scorecards.
- **Every score ships with a `reason` and `evidence_turn_ids`** — a score you cannot audit is a vibe. Evidence pointers let anyone open the transcript and check the judge.
- **Disk cache keyed by (call_id, dimension, prompt_hash)** — rerunning the pipeline is free, and identical inputs cannot silently produce different histories.
- **Disclosed** — we say it is Gemini Flash, on a slide. Hidden judges are how demos lose rooms.

## The bias list (know these cold — engineers will probe)
LLM judges systematically prefer: **longer answers** (verbosity bias), **the first option shown** (position bias), **their own model family's style** (self-preference), and they **drift lenient** without anchors. Mitigations we use: anchored scales (defined 0/0.5/1 meanings, not "rate 1–10"), required evidence, temperature 0, and human calibration. Notebook 04 is the teeth.

In [ ]:
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "rubric.yaml").exists())
import sys
sys.path.insert(0, str(ROOT / "pipeline"))
print("repo root:", ROOT)

import json
from judge import get_client, judge_dimension, judge_config

client = get_client()
print("judge model:", judge_config()[0])
call = json.loads((ROOT / "data" / "normalized" / "swz_MUL0035.json").read_text())
snippet = "\n".join(f"{t['turn_id']} {t['speaker']}: {t['text']}" for t in call["turns"][:8])
print(snippet[:600])

Real turns from a real call. Now we judge `repair_quality` — how well the agent handles unclear/partial answers. **PREDICT:** roughly what score does this exchange deserve, and which turn ids should the evidence cite? Commit to numbers before running.

In [ ]:
PROMPT = (
  "You are a strict but fair judge of voice-agent calls. Score ONE dimension.\n\n"
  "Dimension: repair_quality - when the caller's answer is unclear, partial or mistaken,\n"
  "does the agent acknowledge what it got and ask one targeted follow-up (1.0),\n"
  "partially acknowledge but ask clumsily (0.5), or ignore/over-demand/derail (0.0)?\n\n"
  "Call turns:\n" + snippet + "\n\n"
  'Return ONLY JSON: {"score": <0, 0.5 or 1>, "reason": "<one falsifiable sentence>", '
  '"evidence_turn_ids": ["..."]}'
)
verdict, cached = judge_dimension(client, "nb03_demo", "repair_quality", PROMPT)
print("from cache:", cached)
print(json.dumps(verdict, indent=2))

Read the `reason`. Is it **falsifiable** — could you check it against the transcript and catch it lying? That property is the entire difference between "AI scored it 0.5" (useless) and an audit trail. Now run the cell again: `from cache: True`, zero cost, identical verdict. That is the cache contract.

## See the cache with your own eyes

In [ ]:
cache_dir = ROOT / "data" / ".judge_cache"
for f in sorted(cache_dir.glob("*.json"))[-4:]:
    print(f.name)
print("\nkey = call_id __ dimension __ sha256(model|prompt)[:16] -> change ONE character of the prompt and it is a different key (cache miss, fresh judgment). Idempotent and honest.")

## Exercise — feel a bad rubric
Run the SAME snippet through a deliberately bad prompt: unanchored 1–10 scale, no evidence required. **PREDICT:** in what ways will the output be worse, not just different?

In [ ]:
BAD = ("Rate the agent's handling of unclear answers from 1-10.\n\nCall turns:\n" + snippet +
       '\n\nReturn ONLY JSON: {"score": <1-10>, "reason": "<short>", "evidence_turn_ids": []}')
bad_verdict, _ = judge_dimension(client, "nb03_demo", "repair_quality_bad", BAD)
print(json.dumps(bad_verdict, indent=2))

Read what actually came back and judge the *contract*, not the model's mood. The score arrived on a 1–10 scale — but what does a "2" MEAN? Without anchors there is no comparable answer: another call's "3" might describe better or worse behavior. The model may still have volunteered a decent reason or even evidence — modern judges often do — but nothing in the contract FORCED it, so a pipeline cannot rely on it arriving. And a human cannot blind-label "2 vs 3" for agreement stats, so calibration (next notebook) is dead on arrival. Bad rubrics fail *structurally* — comparability, enforceability, calibratability — even when a strong model papers over them on an easy case.

One more thing you just saw: "depart from courage", "head to lift" — that is genuine ASR output mangling place names. Garbled entities are the genre. Judges must reason through transcript noise, and good ones cite it (ours did, in its reason).

## Self-check
1. Why temperature 0 — what claim does it buy us?
2. Recite the cache key and explain why prompt is part of it.
3. Name three judge biases and one mitigation each.
4. Why is "0 / 0.5 / 1 with defined meanings" better than "1–10" for our use?
5. A founder asks: "so the AI grades the AI — why trust it?" Your one-sentence answer?

<details><summary>Answers</summary>

1. Determinism: same call + same rubric → same verdict, so reruns are reproducible and diffs mean something changed in the inputs, not the dice.
2. (call_id, dimension, sha256(model|prompt)) — if the prompt or model changes, old verdicts must not masquerade as current ones.
3. Verbosity → anchored scales & length-blind criteria; position → we judge single transcripts, not A-vs-B orderings; leniency drift / self-preference → human calibration with blind labels (nb 04).
4. Each anchor is a checkable claim; humans can blind-label the same 3 options, enabling agreement stats. Ten unanchored points produce noise no kappa can rescue.
5. "We do not ask you to trust it — we measured where it agrees with blind human labels and we show you the two cases where it was wrong" (that is notebook 04 and Block 7).
</details>